In [0]:
# Databricks notebook source
# MAGIC %pip install azure-storage-blob
# MAGIC %pip install protobuf==3.17.2
# MAGIC %pip install /dbfs/FileStore/TigerML_045/tigerml.core-0.4.5-py3-none-any.whl --no-deps
# MAGIC %pip install holoviews==1.14.9
# MAGIC %pip install python-slugify==3.0.4

# COMMAND ----------

from pyspark.sql.functions import max, col
import pandas as pd
from plotly.subplots import make_subplots
import plotly.graph_objs as go
from pyspark.sql import functions as F
import time
from datetime import datetime
from requests.structures import CaseInsensitiveDict
from monitoring.utils.secret_mapping import SECRET_MAPPING, CONFIGS
import requests
from monitoring.utils import utils
import traceback

# COMMAND ----------

def convert_timestamp(timestamp_ms):
    # Convert milliseconds to seconds
    timestamp_seconds = timestamp_ms / 1000
    # Convert to readable datetime format
    readable_datetime = datetime.fromtimestamp(timestamp_seconds).strftime('%d %B %Y %H:%M:%S')
    return readable_datetime

def get_sliced_df(df, monitoring_type, monitoring_sub_type=None):
    if monitoring_sub_type is None:
        sliced_df = df.filter((col("run_id") == run_id) & (col("monitoring_type") == monitoring_type))
    else:
        sliced_df = df.filter((col("run_id") == run_id) & (col("monitoring_type") == monitoring_type) & (col("monitoring_sub_type") == monitoring_sub_type))
    return sliced_df

# COMMAND ----------

def generate_heatmap(df, value_column='p_val'):
    # Replace NaN values with 0 in the value_column
    df = df.withColumn(value_column, F.when(F.isnan(col(value_column)), 0).otherwise(col(value_column)))

    # Convert to Pandas DataFrame
    pandas_df = df.toPandas()

    # Pivot the DataFrame for heatmap
    heatmap_data = pandas_df.pivot(index="feature_attribute", columns="granularity", values=value_column)

    # Get the unique value of monitoring_algo_name
    monitoring_algo_name = df.select("monitoring_algo_name").distinct().collect()[0][0]

    # Generate heatmap using Plotly
    fig = go.Figure(data=go.Heatmap(
        z=heatmap_data.values,
        x=heatmap_data.columns,
        y=heatmap_data.index,
        text=heatmap_data.values,
        colorbar=dict(title=f'{value_column.capitalize()} ({monitoring_algo_name})')
    ))

    fig.update_layout(
        title=f'Heatmap of Granularities and Feature Attributes ({value_column.capitalize()} Values)',
        xaxis_title='Granularities',
        yaxis_title='Feature Attributes'
    )

    fig.update_layout(
            margin=dict(l=0, r=0, t=0, b=0),  # Reduced margins to minimal values
            autosize=True,
            height=400,  # Adjust height as needed
            width=1350,  # Adjust width as needed to ensure edge-to-edge appearance
            legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1)
        )

    return {"heatmap": fig}

# COMMAND ----------

def generate_line_charts(df, value_column='p_val'):
    # Get unique feature attributes and monitoring_algo_name
    feature_attributes = df.select("feature_attribute").distinct().rdd.flatMap(lambda x: x).collect()
    monitoring_algo_name = df.select("monitoring_algo_name").distinct().collect()[0][0]

    # Dictionary to store the plots
    line_charts = {}

    # Iterate through each feature attribute
    for feature_attribute in feature_attributes:
        # Filter DataFrame for the current feature attribute
        filtered_df = df.filter(df["feature_attribute"] == feature_attribute)

        # Replace NaN values with 0 in the value_column
        filtered_df = filtered_df.withColumn(value_column, F.when(F.isnan(col(value_column)), 0).otherwise(col(value_column)))

        # Convert to Pandas DataFrame
        pandas_df = filtered_df.toPandas()

        # Create line chart using Plotly
        fig = go.Figure()

        fig.add_trace(go.Scatter(
            x=pandas_df["granularity"],
            y=pandas_df[value_column],
            mode='lines+markers',
            name=f'{feature_attribute}',
            line=dict(shape='spline')  # Use 'spline' for curved lines
        ))

        fig.update_layout(
            title=f'Line Chart for Feature Attribute: {feature_attribute}',
            xaxis_title='Granularities',
            yaxis_title=f'{value_column.capitalize()} ({monitoring_algo_name})',
            xaxis=dict(tickangle=45),
            template="plotly_white"
        )

        fig.update_layout(
            margin=dict(l=0, r=0, t=0, b=0),  # Reduced margins to minimal values
            autosize=True,
            height=400,  # Adjust height as needed
            width=1050,  # Adjust width as needed to ensure edge-to-edge appearance
            legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1)
        )

        # Store the plot in the dictionary
        line_charts[feature_attribute] = fig

    return line_charts

# COMMAND ----------


def generate_timeseries_charts(df, monitoring_type, monitoring_sub_type=None, value_column='p_val'):
    # Filter DataFrame based on monitoring_type and monitoring_sub_type
    if monitoring_sub_type is None:
        filtered_df = df.filter((df["monitoring_type"] == monitoring_type))
    else:
        filtered_df = df.filter((df["monitoring_type"] == monitoring_type) & (df["monitoring_sub_type"] == monitoring_sub_type))

    # Group by run_id and granularity, pivot feature_attribute, and select value_column
    pivot_df = filtered_df.groupBy("run_id", "granularity","timestamp") \
                          .pivot("feature_attribute") \
                          .agg(F.first(value_column))

    # Collect all granularities
    granularities = pivot_df.select("granularity").distinct().rdd.flatMap(lambda x: x).collect()
    monitoring_algo_name = df.select("monitoring_algo_name").distinct().collect()[0][0]
    timeseries_dict = {}

    # Plot for each granularity
    for granularity in granularities:
        # Filter DataFrame for the current granularity
        granularity_df = pivot_df.filter(pivot_df["granularity"] == granularity)

        # Create a subplot for each granularity
        fig = go.Figure()
        
        # Plot each feature attribute
        for i, col in enumerate(pivot_df.columns[3:]):  # Skip run_id, granularity, timestamp
            y_values, timestamps = zip(*[(y, t) for y, t in granularity_df.select(col, "timestamp").orderBy("timestamp").collect() if y is not None])
            y_values = list(y_values)
            timestamps = list(timestamps)
            readable_datetimes = [convert_timestamp(ts) for ts in timestamps]
            fig.add_trace(
                go.Scatter(x=readable_datetimes,
                           y=y_values,
                           name=col,
                           mode='lines+markers',
                           line=dict(shape='spline'),
                           visible=True if i == 0 else 'legendonly')
                        )

        # Update layout
        fig.update_layout(
            xaxis_title="Datetime",
            yaxis_title=f'{value_column.capitalize()} ({monitoring_algo_name})',
            margin=dict(l=0, r=0, t=50, b=50),
            autosize=True,
            height=400,
            width=1350,
            legend=dict(
                orientation="h",
                yanchor="top",
                y=-0.3,
                xanchor="center",
                x=0.5
            )
        )

        timeseries_dict[granularity] = fig
    return timeseries_dict

# COMMAND ----------

def get_monitoring_charts(dd_path=None, pd_path=None):
    if dd_path:
        try:
            dd_df = spark.read.format("delta").load(dd_path)
        except Exception as e:
            print(f"Error loading DataFrame from {dd_path}: {e}")
    
    if pd_path:
        try:
            pd_df = spark.read.format("delta").load(pd_path)
        except Exception as e:
            print(f"Error loading DataFrame from {pd_path}: {e}")
    
    # FD charts
    fd_report_dict = {}
    try:
        monitoring_type = "feature_drift"
        monitoring_sub_type = "feature_level"
        fd_df = get_sliced_df(dd_df, monitoring_type, monitoring_sub_type)
        fd_report_dict.update({"Feature Drift Heatmap": generate_heatmap(fd_df)})
        # fd_report_dict.update({"Feature Drift Line Charts": generate_line_charts(fd_df)})
        fd_report_dict.update({"Feature Drift Time Series": generate_timeseries_charts(dd_df, monitoring_type, monitoring_sub_type)})
    except:
        traceback.print_exc()
        pass

    # TD charts
    # tda_report_dict = {}
    # try:
    #     monitoring_type = "target_drift"
    #     monitoring_sub_type = "actual"
    #     tda_df = get_sliced_df(dd_df, monitoring_type, monitoring_sub_type)
    #     tda_report_dict.update({"Target Drift Heatmap": generate_heatmap(tda_df)})
    #     # tda_report_dict.update({"Target Drift Line Charts": generate_line_charts(tda_df)})
    #     tda_report_dict.update({"Target Drift Time Series": generate_timeseries_charts(dd_df, monitoring_type, monitoring_sub_type)})
    # except:
    #     pass

    tdp_report_dict = {}
    try:
        monitoring_type = "target_drift"
        monitoring_sub_type = "predicted"
        tdp_df = get_sliced_df(dd_df, monitoring_type, monitoring_sub_type)
        tdp_report_dict.update({"Target Drift Heatmap": generate_heatmap(tdp_df)})
        # tdp_report_dict.update({"Target Drift Line Charts": generate_line_charts(tdp_df)})
        tdp_report_dict.update({"Target Drift Time Series": generate_timeseries_charts(dd_df, monitoring_type, monitoring_sub_type)})
    except:
        traceback.print_exc()
        pass
    
    # # CD charts
    # cd_report_dict = {}
    # try:
    #     monitoring_type = "concept_drift"
    #     cd_df = get_sliced_df(dd_df, monitoring_type)
    #     cd_report_dict.update({"Concept Drift Heatmap": generate_heatmap(cd_df)})
    #     cd_report_dict.update({"Concept Drift Line Charts": generate_line_charts(cd_df)})
    #     cd_report_dict.update({"Concept Drift Time Series": generate_timeseries_charts(dd_df, monitoring_type)})
    # except:
    #     pass

    # PD charts
    pd_report_dict = {}
    try:
        monitoring_type = "performance_drift"
        value_column = "value"
        sliced_pd_df = get_sliced_df(pd_df, monitoring_type)
        pd_report_dict.update({"Performance Drift Heatmap": generate_heatmap(sliced_pd_df, value_column)})
        pd_report_dict.update({"Performance Drift Line Charts": generate_line_charts(sliced_pd_df, value_column)})
        pd_report_dict.update({"Performance Drift Time Series": generate_timeseries_charts(pd_df, monitoring_type, value_column=value_column)})
    except:
        traceback.print_exc()
        pass

    return {"FeatureDrift": fd_report_dict, "TargetDriftPredicted": tdp_report_dict, "PerformanceDrift": pd_report_dict}

# COMMAND ----------

def __upload_blob_to_azure(dbutils, container_name, blob_path, target_path):
    from azure.storage.blob import BlobServiceClient

    try:
        connection_string = fetch_secret_from_dbutils(
            dbutils, "az-api-storage-connection-string"
        )
        # Initialize the BlobServiceClient using the connection string
        blob_service_client = BlobServiceClient.from_connection_string(
            connection_string
        )

        # Get the container client for the blob object
        container_client = blob_service_client.get_container_client(
            container=container_name
        )

        # Upload the blob file
        with open(blob_path, "rb") as data:
            blob_client = container_client.get_blob_client(target_path)
            blob_client.upload_blob(data)

    except Exception as e:
        # utils.log(f"Error came while uploading blob object from azure : {e}",message_run)
        raise e

def __upload_blob_to_gcp(dbutils, container_name, blob_path, target_path):
    from google.cloud import storage

    try:
        credentials = get_gcp_auth_credentials(dbutils)
        project = fetch_secret_from_dbutils(dbutils, "gcp-api-quota-project-id")
        print(f"GCS project : {' '.join(project)}")
        print(f"GCS Container : {' '.join(container_name)}")
        # Use the obtained credentials to create a client to interact with GCP services
        storage_client = storage.Client(credentials=credentials, project=project)

        bucket_client = storage_client.bucket(container_name)

        # Upload the model file to GCS
        blob = bucket_client.blob(target_path)
        blob.upload_from_filename(blob_path)

    except Exception as e:
        # utils.log(f"Error came while uploading blob object from gcp : {e}",message_run)
        raise e

def upload_blob_to_cloud(**kwargs):
    """
    Upload the blob from the cloud storage.

    This function will help upload the blob from the cloud storage service like Azure, AWS, GCP.

    Parameters
    ----------
    **kwargs : dict
        Keyword arguments containing operation details, including the `resource_type`.

    Returns
    -------
    The result of the dispatched operation based on the `resource_type`.

    Notes
    -----
    - The function loads the blob object from cloud storage with below parameters :
    - For Azure
        - 'dbutils': The dbutils object to retrive the secrets needed for the APIs.
        - 'container_name': The container where the blob object is stored.
        - 'blob_path': The local file path where the blob is present.
        - 'target_path' : The target path where the blob has to be downloaded.

    - For GCP
        - 'dbutils': The dbutils object to retrive the secrets needed for the APIs.
        - 'container_name': The bucket where the blob  object is stored.
        - 'blob_path': The local file path where the blob is present.
        - 'target_path' : The target path where the blob has to be downloaded.

    - It is essential to provide the correct `resource_type`. Currently supported resources are : az, gcp
    """
    resource_type = kwargs.get("resource_type", None)
    if not resource_type or resource_type in [""]:
        raise Exception("Resource type is not passed or is empty.")

    del kwargs["resource_type"]  # Delete the key since it will not be used by modules

    if resource_type not in ["az", "gcp","azure"]:
        raise Exception(f"Uploading blob object from {resource_type} is not supported.")

    if resource_type.lower() in ["az","azure"]:
        return __upload_blob_to_azure(**kwargs)

    if resource_type.lower() == "gcp":
        # utils.log("upload_blob_to_cloud is called",message_run)
        return __upload_blob_to_gcp(**kwargs)

def fetch_secret_from_dbutils(dbutils, key_name):
    _, vault_scope = get_env_vault_scope()
    return dbutils.secrets.get(scope=vault_scope, key=key_name)

def get_env_vault_scope():
    """
    Returns env and vault scope
    """
    import json
    env = (
        dbutils.notebook.entry_point.getDbutils()
        .notebook()
        .getContext()
        .notebookPath()
        .get()
    ).split("/")[2]
    try:
        if len(dbutils.fs.ls('dbfs:/FileStore/jars/MLCORE_INIT/vault_check.json')) == 1:
            # if env == "qa":
            #     with open("/dbfs/FileStore/jars/MLCORE_INIT/vault_check_qa.json", "r") as file:
            #         vault_check_data = json.loads(file.read())
            # else:
            with open("/dbfs/FileStore/jars/MLCORE_INIT/vault_check.json", "r") as file:
                vault_check_data = json.loads(file.read())
            if "@" in env:
                return "qa", vault_check_data['client_name']
            return env, vault_check_data['client_name']
        else:
            return env, vault_scope
    except:
        return env, vault_scope

message_run=[]

def fetch_secrets_from_dbutils(dbutils, message_logs=[]):
    _, vault_scope = get_env_vault_scope()
    secrets_object = {}
    for secret_key, secret_value in SECRET_MAPPING.items():
        try:
            secrets_object[secret_key] = dbutils.secrets.get(
                scope=vault_scope, key=secret_value
            )
        except Exception as e:
            
            # utils.log(
            #     f"Error fetching secret for '{secret_value} using dbutils': {e}",
            #     message_logs,
            #     "warning",
            # )

            # fetch the secret from config file
            secrets_object[secret_key] = CONFIGS.get(secret_value, "")

    return secrets_object
secrets_object = fetch_secrets_from_dbutils(dbutils, message_run)


try:
    API_ENDPOINT = dbutils.widgets.get("tracking_base_url")
except:
    API_ENDPOINT = get_app_url()

env, vault_scope = get_env_vault_scope()


def get_access_tokens(client_id, scope, client_secret, vault_scope):
    """
    Returns a bearer token
    """

    headers = CaseInsensitiveDict()
    headers["Content-Type"] = "application/x-www-form-urlencoded"
    data = {}
    data["client_id"] = client_id
    data["grant_type"] = "client_credentials"
    data["scope"] = scope
    data["client_secret"] = client_secret
    tenant_id = secrets_object.get("az-directory-tenant", "")
    url = "https://login.microsoftonline.com/" + tenant_id + "/oauth2/v2.0/token"
    resp = requests.post(url, headers=headers, data=data).json()
    token = resp["access_token"]
    token_string = "Bearer" + " " + token
    return token_string

def get_headers(vault_scope):
    """
    Returns API headers
    """
    h1 = CaseInsensitiveDict()
    client_id = secrets_object.get("az-api-client-id", "")
    scope = client_id + "/.default"
    client_secret = secrets_object.get("az-api-client-secret", "")
    h1["Authorization"] = get_access_tokens(
        client_id, scope, client_secret, vault_scope
    )
    h1["Content-Type"] = "application/json"
    return h1
    

sdk_session_id = dbutils.widgets.get("sdk_session_id")
job_id = dbutils.widgets.get("monitor_job_id")
run_id = dbutils.widgets.get("monitor_run_id")
dd_path = dbutils.widgets.get("dd_path")
pd_path = dbutils.widgets.get("pd_path")
env = dbutils.widgets.get("env")
project_id = dbutils.widgets.get("project_id")
version = dbutils.widgets.get("version")

MEDIA_ARTIFACTS_ADD = "mlapi/add_media_artifacts"


container_name = "mlcore"
def media_artifacts_add(cloud_provider,target_path,model_artifact_id ="",entity_type=""):

    ts = int(time.time() * 1000000)

    artifacts_data = {
        "project_id": project_id,
        "version": version,
        "job_id": job_id,
        "run_id": run_id,
        "folder_type": "reports",
        "sub_folder_path": "Monitor_Report",
        "media_artifacts_path": target_path,
        "model_artifact_id": model_artifact_id,
        "entity_type": entity_type,
    }

    # Add additional parameters based on cloud provider
    if cloud_provider.lower() == 'azure':
        artifacts_data["container_name"] = container_name
        artifacts_data["az_storage_account"] = secrets_object.get("az-storage-account", "")

    elif cloud_provider.lower() == 'gcp':
        artifacts_data["container_name"] = container_name
        artifacts_data["gcp_project_id"] = quota_project_id

    elif cloud_provider.lower() == 'databricks_uc':
        catalog_details = get_catalog_details(deployment_env).json().get("data")[0]
        artifacts_data["catalog_name"] = catalog_details["catalog_name"]
        artifacts_data["schema_name"] = catalog_details["catalog_schema_name"]
        artifacts_data["volume_name"] = catalog_details["volume_name"] 

    h1 = get_headers(vault_scope)
    response = requests.post(
        API_ENDPOINT + MEDIA_ARTIFACTS_ADD, json=artifacts_data, headers=h1
    )
    utils.log(
        f"\n\
    Logging task:\n\
    endpoint - {MEDIA_ARTIFACTS_ADD}\n\
    status   - {response}\n\
    response - {response.text}\n\
    payload  - {artifacts_data}\n",
        message_run,
    )

    t = str(int(time.time() * 1000000))

    return response

# COMMAND ----------

report_dict = {}
report_dict = get_monitoring_charts(dd_path = dd_path, pd_path = pd_path)
report_dict


# COMMAND ----------

from tigerml.core.reports import create_report

file_path = f"/dbfs/mnt/FileStore/{sdk_session_id}/monitor_report"
report_name = f"monitor_report_{job_id}_{run_id}"
create_report(
    report_dict,
    name=report_name,
    path=file_path,
    format=".html",
    columns=1,
)
report_path = f"{file_path}/{report_name}"
report_directory = f"{env}/media_artifacts/{project_id}/{version}/{job_id}/{run_id}/Monitor_Report"
upload_blob_to_cloud(
    container_name='mlcore',
    blob_path=f"{report_path}.html",
    dbutils=dbutils,
    target_path=f"{report_directory}/{report_name}.html",
    resource_type='az',
)

# COMMAND ----------

media_artifacts_add(cloud_provider="azure",target_path=f"{report_directory}/{report_name}.html", entity_type="monitor_custom_reports")